In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab-mistral/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · The turn: budgets, checkpoints and idempotency

**What you'll learn.** Why the unit of work is a *turn*, how a turn ends no matter what the model does
(budgets), and how a turn survives the pod dying half-way (checkpoints + idempotent writes + at-least-once
redelivery from the queue). The loop is `scalelab/loop.py` — about 60 lines of logic — and it is the same
whether the model call goes to Mistral's API or to a vLLM replica in the customer's cluster.

> **In a design review.** *"Every step is checkpointed in Postgres before I act on its result; writes are keyed by turn,
> step and arguments in Redis; so the queue can redeliver freely and a redelivered turn gets the ticket it already
> created back instead of a second one. Graceful failures are acknowledged, not retried — retrying them only
> spends money."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. Read the loop

In [ ]:
from scalelab.loop import run_turn, Budget, Store, execute_tool
from scalelab.model import FakeModel
from scalelab.tools import Tools
print(inspect.getsource(run_turn))

## 2. One turn, end to end

The fake model issues tool calls for the intent it detects, then answers. Watch the steps and the checkpoints.

In [ ]:
model, tools, store = FakeModel(), Tools(), Store()
r = await run_turn("turn-1", "my bill is higher than usual this month", model=model, tools=tools, store=store)
print(f"{r.status}  steps={r.steps}  tools={r.tool_names}  tokens={r.tokens}  cost=${r.cost_usd:.4f}  latency={r.latency_s:.1f}s  served by the API: {r.hosted_calls} calls")
for step in store.get_steps("turn-1"):
    payload = step["payload"]
    print(step["index"], step["kind"], payload.get("tool_calls") or payload.get("tools"), "" if step["kind"] == "tool" else f"{payload['input_tokens']} in ({payload['cached_tokens']} cached) / {payload['output_tokens']} out")

## 3. Budgets: the loop must always terminate

Steps, tokens, dollars and wall-clock are independent dimensions — each fails differently. A turn that runs out ends *gracefully* with an apology, and that outcome is terminal.

In [ ]:
r = await run_turn("turn-2", "my bill is wrong", model=model, tools=tools, store=store, budget=Budget(max_steps=2))
print(r.status, r.error, "| steps taken:", r.steps, "|", r.text)

## 4. Crash, redeliver, resume

We kill the pod right after the ticket has been created *and checkpointed*. On Kubernetes this is the
`terminationGracePeriodSeconds` after SIGTERM during a rollout or a node drain, an OOM kill, or a nack. The queue
redelivers; the loop replays the checkpointed steps and only executes what is missing. The write is keyed
`{turn}:{step}:{i}:{tool}`, so even a replay that reached the tool would be deduplicated.

In [ ]:
store2, tools2 = Store(), Tools()

class PodKilled(Exception):
    pass

original = store2.append_step
def crash_after_ticket(turn_id, step):
    original(turn_id, step)
    if step["kind"] == "tool" and "create_ticket" in step["payload"]["tools"]:
        raise PodKilled("SIGTERM")
store2.append_step = crash_after_ticket

try:
    await run_turn("turn-3", "I want to complain and speak to a human", model=model, tools=tools2, store=store2)
except PodKilled:
    print("pod died with", len(store2.get_steps("turn-3")), "checkpointed steps; tickets so far:", tools2.tickets)

store2.append_step = original            # the redelivery lands on a healthy pod
r = await run_turn("turn-3", "I want to complain and speak to a human", model=model, tools=tools2, store=store2)
print(f"redelivery: {r.status}, resumed {r.resumed_steps} steps, {r.model_attempts} new model call, tickets: {tools2.tickets}")
print("idempotency markers:", list(store2.idem))

## 5. The queue contract (RabbitMQ quorum queue)

| Outcome | Consumer does | The queue does | Why |
|---|---|---|---|
| completed | `ack` | delete | done |
| failed gracefully (budget, overload) | `ack` | delete | the user got a message; retrying spends more money on a turn they gave up on |
| transient infrastructure trouble (Postgres down, session locked) | `nack` (requeue) | redeliver after the consumer's backoff; `x-delivery-count` grows | the checkpoint makes redelivery safe |
| `delivery-limit` (5) reached | — | dead-letter exchange → `agent-turns.dlq` | alert a human; replay tooling |

The consumer prefetch bounds how many turns a pod holds; the turn budget (45 s) sits well under any consumer timeout.
Quorum queues give at-least-once delivery; design for it — exactly-once is the idempotency key, not the broker.

## Your turn — solutions

#### (a) Idempotency key

A stable key from turn id, step index, call index, tool name and a hash of the arguments (sorted JSON).

In [ ]:
import hashlib, json
def idempotency_key(turn_id, step, i, tool, args):
    digest = hashlib.sha256(json.dumps(args, sort_keys=True).encode()).hexdigest()[:16]
    return f"{turn_id}:{step}:{i}:{tool}:{digest}"

In [ ]:
def _a():
    k1 = idempotency_key("t1", 3, 0, "create_ticket", {"category": "billing", "summary": "x"})
    k2 = idempotency_key("t1", 3, 0, "create_ticket", {"summary": "x", "category": "billing"})
    k3 = idempotency_key("t1", 3, 0, "create_ticket", {"category": "network"})
    assert k1 == k2 and k1 != k3 and k1.startswith("t1:3:0:create_ticket:")
check("a: idempotency key", _a)

#### (b) Replay decision

Given the checkpointed steps and the planned steps of a turn, return the indices that still have to be executed.

In [ ]:
def steps_to_run(checkpointed: list[dict], planned: list[str]) -> list[int]:
    done = {c["index"] for c in checkpointed}
    return [i for i in range(len(planned)) if i not in done]

In [ ]:
def _b():
    assert steps_to_run([{"index": 0, "kind": "model"}, {"index": 1, "kind": "tool"}], ["model", "tool", "model"]) == [2]
    assert steps_to_run([], ["model"]) == [0]
    assert steps_to_run([{"index": 0, "kind": "model"}], ["model"]) == []
check("b: replay decision", _b)

#### (c) Ack or nack

Classify an exception: 'ack' for terminal outcomes (the user got an answer, or never will), 'nack' for transient infrastructure trouble.

In [ ]:
class StoreUnavailable(Exception): pass
class SessionLocked(Exception): pass
from scalelab.loop import BudgetExceeded
from scalelab.resilience import RateLimited
from scalelab.model import ServerOverloaded

def ack_or_nack(exc: Exception) -> str:
    return "nack" if isinstance(exc, (StoreUnavailable, SessionLocked)) else "ack"

In [ ]:
def _c():
    assert ack_or_nack(BudgetExceeded("steps")) == "ack"       # graceful failure is terminal
    assert ack_or_nack(RateLimited()) == "ack"                 # the loop already retried within the deadline
    assert ack_or_nack(ServerOverloaded()) == "ack"            # a 503 from a capped vLLM queue is the same story
    assert ack_or_nack(StoreUnavailable()) == "nack"
    assert ack_or_nack(SessionLocked()) == "nack"
check("c: ack or nack", _c)

#### (d) Budget check

Return the first exceeded dimension ('steps', 'tokens', 'cost', 'deadline') or None.

In [ ]:
def check_budget(steps, tokens, cost_usd, seconds_left, budget: Budget):
    if steps >= budget.max_steps: return "steps"
    if tokens >= budget.max_tokens: return "tokens"
    if cost_usd >= budget.max_cost_usd: return "cost"
    if seconds_left <= 1: return "deadline"
    return None

In [ ]:
def _d():
    b = Budget()
    assert check_budget(2, 5000, 0.01, 40, b) is None
    assert check_budget(6, 5000, 0.01, 40, b) == "steps"
    assert check_budget(2, 50_000, 0.01, 40, b) == "tokens"
    assert check_budget(2, 5000, 0.30, 40, b) == "cost"
    assert check_budget(2, 5000, 0.01, 0.5, b) == "deadline"
check("d: budget check", _d)

## Takeaways for the conversation

- The turn is the unit of work, cost and idempotency; the step is the unit of checkpointing.
- Checkpoint before you act on a result; key writes by turn/step/args; then redelivery is free.
- Budgets on four dimensions plus a deadline; a budget failure is a *terminal*, acknowledged outcome. On a self-hosted fleet the cost dimension is tokens, not dollars — the GPUs are paid for — but the deadline matters more, because a saturated replica does not say no.
- Ack graceful failures, nack only transient infrastructure trouble, dead-letter after a few attempts.

## Verify before the conversation

RabbitMQ quorum-queue semantics (`delivery-limit`, `x-delivery-count`, dead-letter exchange); the pod's `terminationGracePeriodSeconds`; Postgres row size for a checkpoint (truncate tool results); whether Mistral's Agents & Conversations API (beta) would replace this loop for the customer.